In [ ]:
pip install torch numpy

In [5]:
import numpy as np
from typing import *


def Quant_Int8(input_embed: np.ndarray):
    # 获取序列的最大值
    max_value = np.max(np.abs(input_embed))

    if max_value < 10**-8:
        return np.clip(np.round(input_embed), -128, 127).astype(np.int8)
    scale = max_value / 127
    return np.clip(np.round(input_embed / scale), -128, 127).astype(np.int8)


# ==================== Example ====================
if __name__ == "__main__":
    # 示例1: 常规浮点数组量化
    print("--- 示例1: 常规量化 ---")
    data_normal = np.array([0.5, -0.3, 0.8, -0.1, 0.0])
    quantized_normal = Quant_Int8(data_normal)
    print(f"原始数据:   {data_normal}")
    print(f"量化结果:   {quantized_normal}")
    print(f"数据类型:   {quantized_normal.dtype}")
    # 预期: scale = 0.8/127 ≈ 0.0063, 0.5/0.0063 ≈ 79

    # 示例2: 接近零的极小值（触发 max_value < 1e-8 的保护分支）
    print("\n--- 示例2: 极小值保护 ---")
    data_tiny = np.array([1e-10, -5e-9, 3e-11])
    quantized_tiny = Quant_Int8(data_tiny)
    print(f"原始数据:   {data_tiny}")
    print(f"量化结果:   {quantized_tiny}")
    # 预期: 全部被 round 后 clip 为 0

    # 示例3: 包含超出 [-128, 127] 映射范围的值（验证 clip 截断）
    print("\n--- 示例3: 边界值截断 ---")
    data_edge = np.array([-1.0, 0.0, 1.0, 0.999])
    quantized_edge = Quant_Int8(data_edge)
    print(f"原始数据:   {data_edge}")
    print(f"量化结果:   {quantized_edge}")
    # 预期: -1.0 → -127, 1.0 → 127, 0.999 → 127

    # 示例4: 二维矩阵量化（验证对多维数组的支持）
    print("\n--- 示例4: 二维矩阵 ---")
    data_2d = np.random.uniform(-0.5, 0.5, size=(3, 4))
    quantized_2d = Quant_Int8(data_2d)
    print(f"原始数据:\n{data_2d}")
    print(f"量化结果:\n{quantized_2d}")
    print(f"形状保持:   {data_2d.shape} -> {quantized_2d.shape}")

--- 示例1: 常规量化 ---
原始数据:   [ 0.5 -0.3  0.8 -0.1  0. ]
量化结果:   [ 79 -48 127 -16   0]
数据类型:   int8

--- 示例2: 极小值保护 ---
原始数据:   [ 1.e-10 -5.e-09  3.e-11]
量化结果:   [0 0 0]

--- 示例3: 边界值截断 ---
原始数据:   [-1.     0.     1.     0.999]
量化结果:   [-127    0  127  127]

--- 示例4: 二维矩阵 ---
原始数据:
[[ 0.23964318 -0.19421152 -0.43297733  0.47297779]
 [ 0.27718338  0.37255441  0.33382487  0.37294087]
 [-0.4338109  -0.03253672  0.2418127  -0.29136117]]
量化结果:
[[  64  -52 -116  127]
 [  74  100   90  100]
 [-116   -9   65  -78]]
形状保持:   (3, 4) -> (3, 4)


In [ ]:
def INT8_FP16(input_embed,scale):
    return input_embed * scale